In [ ]:
!pip install -U mplsoccer
!pip install statsbombpy

In [ ]:
from statsbombpy import sb
import pandas as pd
from mplsoccer import Pitch,Sbopen
import matplotlib.pyplot as plt

In [ ]:
comps = sb.competitions()
comps['competition_name'].unique()

In [ ]:
comps[comps['competition_name']=='African Cup of Nations']

In [ ]:
matches = sb.matches(competition_id=1267, season_id=107)
matches[matches['home_team']=='Nigeria']

In [ ]:

event = sb.events(match_id=3923881)
event

In [ ]:
parser = Sbopen()
df, related,freeze,tactics = parser.event(3923881)

In [ ]:
df.columns

In [ ]:
df = df[df['team_name']=='Nigeria']
df

In [ ]:
df['type_name'].unique()

In [ ]:
passes = df[df['type_name']=='Pass']
passes

In [ ]:
passes = passes[['id','minute','player_id','player_name','x','y','end_x', 'end_y','pass_recipient_id','pass_recipient_name','outcome_id','outcome_name']]
passes

In [ ]:
passes['outcome_name'].unique()

In [ ]:
successful = passes[passes['outcome_name'].isnull()]
successful

In [ ]:
subs = df[df['type_name']=='Substitution']
subs = subs['minute']
firstSub= subs.min()
firstSub

In [ ]:
successful = successful[successful['minute']<firstSub]
successful

In [ ]:
df_lineup = parser.lineup(3923881)
jersey_data =df_lineup[['player_id','jersey_number']]
jersey_data

In [ ]:
successful = pd.merge(successful,jersey_data,on='player_id')
successful.rename(columns={'jersey_number':'passer'},inplace=True)
successful

In [ ]:
jersey_data.rename(columns={'player_id':'pass_recipient_id'},inplace=True)
successful = pd.merge(successful,jersey_data,on='pass_recipient_id')
successful.rename(columns={'jersey_number':'recipient'},inplace=True)
successful

In [ ]:
average_locations = successful.groupby('passer').agg({'x':['mean'],'y':['mean','count']})

In [ ]:
average_locations.columns = ['x','y','count']
average_locations

In [ ]:
pass_between = successful.groupby(['passer','recipient']).id.count().reset_index()
pass_between.rename(columns={'id':'pass_count'},inplace=True)
pass_between

In [ ]:
pass_between = pd.merge(pass_between, average_locations,on='passer')
pass_between

In [ ]:
average_locations=average_locations.rename_axis('recipient')
pass_between = pd.merge(pass_between, average_locations,on='recipient',suffixes=['','_end'])
pass_between

In [ ]:
pass_between = pass_between[pass_between['pass_count']>1]

In [ ]:
pass_between.head()

In [ ]:
pitch = Pitch(pitch_color='#aabb97', line_color='white',
              stripe_color='#c2d59d', stripe=True)  

fig,ax = pitch.draw(figsize=(8,6))

#Draw arrows and nodes
pass_lines = pitch.lines(1.2*pass_between.x, 0.8*pass_between.y,
                         1.2*pass_between.x_end, 0.8*pass_between.y_end, lw=pass_between.pass_count*0.5,
                         color='red', zorder=1, ax=ax)

nodes = pitch.scatter(1.2*average_locations.x,0.8*average_locations.y,s=20*average_locations['count'].values,color='white',edgecolors='#a6aab3',linewidth=1,ax=ax)


# Put jersey number in the nodes

for index,row in average_locations.iterrows():
    pitch.annotate(index,xy=(1.2*row.x,0.8*row.y),c='#161A30', fontweight='bold',va='center',ha='center',size=8, ax=ax)

    
ax.set_title('Nigeria vs Ghana',color='red',va='center',ha='center',fontsize=12,fontweight='bold',pad=20)

ax.annotate('African Cup of Nations Final', xy=(0.5, 1), xytext=(0, 0),
             xycoords='axes fraction', textcoords='offset points',
             fontsize=10, color='#0E2954', va='top', ha='center')

ax.text(102, 85, '@PassMapProject', color='#0E2959', va='bottom', ha='center', fontsize=10)